In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from feature_engine.encoding import MeanEncoder
from feature_engine.selection import DropCorrelatedFeatures, DropDuplicateFeatures

from utils import Pipeline, Pipe, BasePipe, Rename

In [9]:
data = pd.read_csv("./data/train.csv", index_col=0)

to_lag = ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]
to_drop = ["E7", "V10", "S3", "M1", "M13", "M14", "M6", "V9"]  #缺失值多，直接删除列

#处理bool列
for i in range(1, 10):
    data[f"D{i}"] = data[f"D{i}"] != 0
    data[f"D{i}"] = data[f"D{i}"].astype("category")

#处理收益率列
for i in to_lag:
    data[f"lag_{i}"] = data[i].shift(1)

data["target"] = (data["forward_returns"] - data["risk_free_rate"]) > 0  #使用指数收益率是否大于无风险利率作为预测值
data = data.drop(to_drop + to_lag, axis=1)
data = data[-5400:]
data

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V3,V4,V5,V6,V7,V8,lag_forward_returns,lag_risk_free_rate,lag_market_forward_excess_returns,target
date_id,,,,,,,,,,,,,,,,,,,,,
3621,False,False,False,True,False,False,False,False,False,1.069307,...,0.114418,0.252646,-0.070909,0.000661,-1.007954,0.000661,0.008454,0.000041,0.008105,True
3622,False,False,True,True,False,False,False,False,False,1.068143,...,0.248677,0.326720,-0.221641,0.000661,-1.081144,0.000661,0.006378,0.000041,0.006029,False
3623,False,False,False,True,False,False,False,False,False,1.066981,...,0.093915,0.316138,-0.011475,0.000661,-0.956609,0.000661,-0.004165,0.000040,-0.004513,True
3624,False,False,False,True,False,False,False,False,False,1.065821,...,0.119048,0.341270,-0.172679,0.000661,-1.023332,0.000661,0.000455,0.000039,0.000108,False
3625,False,False,False,False,False,False,False,False,False,1.064664,...,0.111111,0.326058,-0.171993,0.000661,-0.766254,0.000661,-0.008542,0.000038,-0.008889,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9016,False,False,False,True,False,False,False,False,False,1.493117,...,0.208995,0.484788,0.717308,0.677249,-0.327455,0.083995,0.010401,0.000152,0.009936,False
9017,False,False,False,True,False,False,False,False,False,1.490889,...,0.082011,0.482804,1.001028,0.596561,-0.372979,0.094246,-0.000015,0.000151,-0.000477,False
9018,False,False,False,True,False,True,False,False,False,1.488667,...,0.334656,0.486772,0.894502,0.656746,-0.282024,0.090608,-0.005199,0.000150,-0.005661,True


In [10]:
train = data[:5000]
test = data[5000:]
x_train, y_train = train.drop("target", axis=1), train["target"]
x_test, y_test = test.drop("target", axis=1), test["target"]
x_train

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V2,V3,V4,V5,V6,V7,V8,lag_forward_returns,lag_risk_free_rate,lag_market_forward_excess_returns
date_id,,,,,,,,,,,,,,,,,,,,,
3621,False,False,False,True,False,False,False,False,False,1.069307,...,0.152778,0.114418,0.252646,-0.070909,0.000661,-1.007954,0.000661,0.008454,0.000041,0.008105
3622,False,False,True,True,False,False,False,False,False,1.068143,...,0.171296,0.248677,0.326720,-0.221641,0.000661,-1.081144,0.000661,0.006378,0.000041,0.006029
3623,False,False,False,True,False,False,False,False,False,1.066981,...,0.160053,0.093915,0.316138,-0.011475,0.000661,-0.956609,0.000661,-0.004165,0.000040,-0.004513
3624,False,False,False,True,False,False,False,False,False,1.065821,...,0.151455,0.119048,0.341270,-0.172679,0.000661,-1.023332,0.000661,0.000455,0.000039,0.000108
3625,False,False,False,False,False,False,False,False,False,1.064664,...,0.130952,0.111111,0.326058,-0.171993,0.000661,-0.766254,0.000661,-0.008542,0.000038,-0.008889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8616,False,False,False,True,False,False,False,False,False,3.746269,...,0.621693,0.638228,0.530423,0.961743,0.478175,-0.760544,0.263558,-0.006867,0.000208,-0.007388
8617,False,False,False,True,False,False,False,False,False,3.731523,...,0.589947,0.301587,0.532407,0.531891,0.610450,-0.770230,0.256944,0.005943,0.000208,0.005422
8618,True,True,False,True,False,True,False,False,False,3.716951,...,0.561508,0.163360,0.503307,0.622461,0.537698,-0.833843,0.237434,0.005557,0.000208,0.005036


In [12]:
#定义Pipe，需要实现fit_transform和transform
class EncodeCategorical(BasePipe):
    def __init__(self):
        self.encoder = MeanEncoder(unseen="encode")
        self.has_category = False

    def fit_transform(self, x_train, y_train):
        self.has_category = x_train.select_dtypes("category").columns.any()

        if not self.has_category:
            return x_train, y_train

        x_new = x_train.copy()
        return self.encoder.fit_transform(x_new, y_train), y_train

    def transform(self, x_test, y_test):
        if not self.has_category:
            return x_test, y_test

        x_new = x_test.copy()
        return self.encoder.transform(x_new), y_test


pipeline = Pipeline([
    Pipe(DropDuplicateFeatures()),  #能够在sklearn.pipeline中使用的，在这里可以套一层Pipe后使用
    Pipe(DropCorrelatedFeatures()),
    Pipe(StandardScaler(), force_df=True),  #强制要求保持原列名和索引
    EncodeCategorical(),  #将类别型特征转换为数值型
    Rename()  #将所有列重命名
])

x_train_impute, y_train_impute = pipeline.fit_transform(x_train, y_train)
x_test_impute, y_test_impute = pipeline.transform(x_test, y_test)

x_train_impute

,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,...,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70
date_id,,,,,,,,,,,,,,,,,,,,,
3621,-0.180641,-0.226503,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,-0.639963,-1.166366,...,-0.456077,0.749272,-1.549957,-1.203526,-0.546884,-1.419709,-0.480377,-1.410220,-1.393507,-0.239700
3622,-0.180641,4.414952,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,-0.641692,-1.165310,...,-0.537130,1.452750,-1.549957,-1.203526,-0.652843,-1.354681,-0.612286,-1.410220,-1.393507,-0.237418
3623,-0.180641,-0.226503,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,-0.643418,-1.164255,...,-0.578549,1.193923,-1.549957,-1.203526,-0.567237,-1.394162,-0.428365,-1.410220,-1.393507,-0.251108
3624,-0.180641,-0.226503,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,-0.645139,-1.163199,...,-0.822760,0.552386,-1.549957,-1.203526,-0.649633,-1.424354,-0.569438,-1.410220,-1.393507,-0.266509
3625,-0.180641,-0.226503,-1.157466,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,-0.646857,-1.162143,...,-0.877940,0.284710,-1.549957,-1.203526,-0.429118,-1.496349,-0.568838,-1.410220,-1.393507,-0.272784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8616,-0.180641,-0.226503,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,3.334779,-0.673355,...,1.211009,1.490358,-0.631308,1.032655,-0.373469,0.226898,0.423319,0.211073,-0.621433,2.162334
8617,-0.180641,-0.226503,0.863956,-0.484951,-0.559795,-0.221083,-0.408153,-0.408153,3.312884,-0.674411,...,1.196985,1.381960,-0.569237,0.983881,-0.370953,0.115421,0.047146,0.660185,-0.640856,2.162334
8618,5.535844,-0.226503,0.863956,-0.484951,1.786369,-0.221083,-0.408153,-0.408153,3.291247,-0.675466,...,1.226221,1.430628,-0.759588,0.807871,-0.408179,0.015557,0.126406,0.413173,-0.698154,2.162334
